# Практическая работа № 7 (пример выполнения)
## ECS, слои и async/threading для «Тайны исчезнувшего дива»

**Курс:** ООП для разработки компьютерных игр
**Модуль:** 7 — Архитектура игры
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине
**Формат:** демонстрационный блокнот с полным разбором задания
**Цель:** показать, как внедрить ECS, разложить код по слоям и добавить async/threading

---

## 📖 Контекст

После модулей 1–6 проект имеет все шесть паттернов, но сущности всё ещё классы-«толстяки», а код лежит в одной куче.

## 🚧 Какие проблемы остались

1. **Данные и логика смешаны.** `Detective` и `Div` — классы с методами и полями.
2. **Код в одной куче.** Нет границ между логикой, отображением и техникой.
3. **Загрузка блокирует.** Синхронная загрузка ассетов замораживает игровой цикл.

## 🎯 Что демонстрирует блокнот

| Тема | Проблема | Улучшение |
|---|---|---|
| **ECS** | Толстые классы | Entity + компоненты + системы |
| **Слои** | Монолит | domain/application/infrastructure/presentation |
| **asyncio** | Блокирующая загрузка | `gather` параллельно |
| **threading** | Блокирующий лог | Фоновый поток + Queue |

## Ключевая идея

Хорошая архитектура — это **границы**: данные отдельно от логики, логика отдельно от техники и отображения, а I/O не блокирует игру.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# Шаг 1. Теория: зачем нужны ECS, слои и async/threading

## 1.1. ECS: данные отдельно от логики

**Проблема:** `Detective` содержит и поля, и методы.

**Решение:** сущность — набор компонентов-данных, системы — логика над ними.

## 1.2. Слои: границы зависимостей

**Проблема:** любой модуль может импортировать любой другой.

**Решение:** слои с зависимостями внутрь (presentation → application → domain).

## 1.3. async/threading: не блокировать игру

**Проблема:** загрузка ассетов замораживает цикл.

**Решение:** asyncio для параллельного I/O, поток для фонового логирования.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html).
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).
- Документация Python: [asyncio](https://docs.python.org/3/library/asyncio.html), [threading](https://docs.python.org/3/library/threading.html).

---
# Шаг 2. Реализация ECS

In [ ]:
from dataclasses import dataclass


@dataclass
class Position:
    x: float = 0.0
    y: float = 0.0


@dataclass
class Velocity:
    dx: float = 0.0
    dy: float = 0.0


@dataclass
class Health:
    hp: int = 100


class Entity:
    _next_id = 0

    def __init__(self):
        self.id = Entity._next_id
        Entity._next_id += 1
        self.components = {}

    def add(self, component):
        self.components[type(component).__name__.lower()] = component
        return self


class MovementSystem:
    def update(self, entities, dt):
        for e in entities:
            pos = e.components.get("position")
            vel = e.components.get("velocity")
            if pos and vel:
                pos.x += vel.dx * dt
                pos.y += vel.dy * dt


class HealthSystem:
    def update(self, entities, dt):
        for e in entities:
            health = e.components.get("health")
            if health and health.hp <= 0:
                print(f"Сущность {e.id} погибла")


# Сборка сущностей
detective = Entity().add(Position(0, 0)).add(Health(100))
bullet = Entity().add(Position(0, 0)).add(Velocity(100, 0))

# Симуляция
movement = MovementSystem()
health = HealthSystem()
for frame in range(3):
    movement.update([detective, bullet], dt=1.0)
    health.update([detective, bullet], dt=1.0)
    print(f"Кадр {frame}: пуля на x={bullet.components['position'].x:.0f}")
print(f"Детектив имеет {len(detective.components)} компонента")

**Что мы видим:** `MovementSystem` двигает сущность с `Position`+`Velocity`, не зная про её «тип». Детектив и пуля — одинаково сущности, различаются только набором компонентов.

---
# Шаг 3. Слоистая архитектура

In [ ]:
# Демонстрация структуры слоёв (без создания реальных файлов)
layers = {
    "domain": ["ecs.py", "components.py", "world_state.py"],
    "application": ["game.py", "game_loop.py", "systems/"],
    "infrastructure": ["audio_manager.py", "config.py", "asset_loader.py", "background.py"],
    "presentation": ["renderer.py", "ui.py", "input_handler.py"],
}

print("Структура слоёв:")
for layer, files in layers.items():
    print(f"  {layer}/")
    for f in files:
        print(f"    {f}")

print()
print("Направление зависимостей:")
print("  presentation -> application -> domain")
print("  infrastructure -> (реализует интерфейсы) domain/application")

**Что мы видим:** код разложен по слоям. domain — чистая логика без I/O, presentation — отображение, application — оркестрация, infrastructure — техника.

---
# Шаг 4. Асинхронная загрузка ассетов (asyncio)

In [ ]:
import asyncio


async def load_asset(path, delay):
    await asyncio.sleep(delay)  # имитация загрузки с диска
    return f"asset:{path}"


async def load_all():
    results = await asyncio.gather(
        load_asset("hero.png", 0.2),
        load_asset("kuzia.png", 0.1),
        load_asset("map.png", 0.3),
    )
    return results


assets = asyncio.run(load_all())
print("Загружено параллельно:")
for a in assets:
    print(" ", a)

**Что мы видим:** три ассета загружаются «одновременно» через `gather`, а не последовательно. Для I/O это даёт реальный выигрыш во времени.

---
# Шаг 5. Фоновый поток (threading + Queue)

In [ ]:
import threading
import queue
import time

log_queue = queue.Queue()


def background_logger(stop_event):
    """Фоновый поток: выводит сообщения из очереди."""
    while not stop_event.is_set() or not log_queue.empty():
        try:
            msg = log_queue.get(timeout=0.1)
        except queue.Empty:
            continue
        print(f"[лог] {msg}")


stop_event = threading.Event()
thread = threading.Thread(target=background_logger, args=(stop_event,), daemon=True)
thread.start()

log_queue.put("игра началась")
log_queue.put("Кузя нашёл улику")
time.sleep(0.3)
stop_event.set()
thread.join(timeout=1)
print("Фоновый поток завершён")

**Что мы видим:** поток читает сообщения из потокобезопасной очереди и завершается по событию. Игровой цикл не блокируется выводом лога.

---
# Шаг 6. Юнит-тесты

## 6.1. Тесты ECS

```python
def test_entity_add_component():
    e = Entity().add(Position(1, 2))
    assert e.components["position"] == Position(1, 2)


def test_movement_moves():
    e = Entity().add(Position(0, 0)).add(Velocity(10, 0))
    MovementSystem().update([e], dt=1.0)
    assert e.components["position"].x == 10.0


def test_movement_ignores_without_velocity():
    e = Entity().add(Position(0, 0))
    MovementSystem().update([e], dt=1.0)
    assert e.components["position"].x == 0.0
```

## 6.2. Тесты загрузчика

```python
def test_load_assets_parallel():
    assets = asyncio.run(load_all())
    assert len(assets) == 3
    assert assets[0] == "asset:hero.png"
```

Запустите `pytest -v` — все тесты должны проходить.

---
# Шаг 7. Документация

## 7.1. ADR-007

```markdown
# ADR-007: ECS, слои, async/threading

## Контекст
- Сущности — толстые классы;
- Код монолитен;
- Загрузка блокирует игру.

## Решение
- ECS: Entity + компоненты + системы;
- Слои: domain/application/infrastructure/presentation;
- asyncio для загрузки, threading для лога.

## Альтернативы
1. Оставить ООП-иерархию — проще, но хуже расширяется.
2. Синхронная загрузка — проще, но блокирует.

## Последствия
- ✅ Данные и логика разделены.
- ✅ Чёткие границы зависимостей.
- ✅ I/O не блокирует игру.
- ⚠️ Потоки требуют дисциплины (только I/O, только Queue).
```

## 7.2. Схема слоёв
Добавьте схему зависимостей слоёв и описание модулей.

## 7.3. PEP 8

```bash
pycodestyle domain/ application/ infrastructure/ presentation/
```

Исправьте замечания.

---
# Итоги

## Что сделали

| Тема | Проблема | Улучшение |
|---|---|---|
| **ECS** | Толстые классы | Entity + компоненты + системы |
| **Слои** | Монолит | 4 слоя с зависимостями внутрь |
| **asyncio** | Блокирующая загрузка | `gather` параллельно |
| **threading** | Блокирующий лог | Поток + Queue |

## Что изменилось в проекте

**До:** классы-«толстяки», монолитный код, блокирующий I/O.

**После:** ECS разделяет данные и логику, слои задают границы, async/threading убирают блокировки.

## Как это подготовит модуль 8

- **Тестирование:** domain без I/O легко тестировать.
- **Рефакторинг:** чёткие границы упрощают изменения.
- **Защита:** структура слоёв — готовая тема для презентации.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html).
- Martin R. C. «Clean Architecture».

**Углублённая:**
- [Sander Mertens: ECS FAQ](https://github.com/SanderMertens/ecs-faq).
- [esper](https://pypi.org/project/esper/) — ECS для Python.
- Документация Python: [asyncio](https://docs.python.org/3/library/asyncio.html), [threading](https://docs.python.org/3/library/threading.html).